# Volleyball tracking

Same pipeline as `main.py`: player tracking with ByteTrack, team assignment from the court, ball tracking, an annotated video, and player tracking CSV.

**Run all cells top to bottom.** Keep the notebook beside `volleyball_sample.mp4`, `yolov8n.pt` (the volleyball ball model), and `court_areas.json`. The standard player model goes in `player_models/yolov8n.pt` and is downloaded if missing. The court selector opens a separate OpenCV window if the saved corners do not match your video.

Install dependencies in the notebook environment if needed: `%pip install "ultralytics>=8.3" opencv-python numpy`.

If Jupyter starts in a different folder, change `project_folder` in the second code cell.

In [1]:
import csv
import json
import math
from collections import deque
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO


Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\akul1\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [2]:
# keep the notebook in the same folder as your video and model
project_folder = Path.cwd()

input_path = project_folder / "volleyball_sample.mp4"
ball_model_path = project_folder / "yolov8n.pt"  # volleyball-trained ball model
player_model_path = project_folder / "player_models" / "yolov8n.pt"  # standard person model
court_settings_path = project_folder / "court_areas.json"
output_folder = project_folder / "outputs"

print("Project folder:", project_folder)
print("Video:", input_path)


Project folder: c:\Users\akul1\OneDrive - Red River College Polytech\3704-Neural Networks and Deep Learning\VsCode\volleyball-analysis-main
Video: c:\Users\akul1\OneDrive - Red River College Polytech\3704-Neural Networks and Deep Learning\VsCode\volleyball-analysis-main\volleyball_sample.mp4


## Court setup

Load the saved corners, or click four corners for each side.

In [3]:
TEAM_COLORS = {
    "A": (0, 220, 255),   # yellow
    "B": (255, 180, 0)    # blue
}


def load_or_select_court(cap, settings_path, video_path):
    # load the saved court or pick corners manually

    settings_path = Path(settings_path)

    success, frame = cap.read()

    if not success:
        raise RuntimeError("Cannot read the court setup frame.")

    # put the video back at frame 0
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

    height, width = frame.shape[:2]

    video_info = {
        "video": str(Path(video_path).resolve()),
        "width": width,
        "height": height
    }

    # only reuse corners from the same video
    if settings_path.exists():
        data = json.loads(settings_path.read_text())

        if data.get("video_info") == video_info:
            print("Loaded saved court areas.")

            return {
                team: np.array(
                    data["areas"][team],
                    dtype=np.float32
                )
                for team in ("A", "B")
            }

    # shrink the selection window if needed
    scale = min(
        1.0,
        1200 / width,
        700 / height
    )

    display_width = round(width * scale)
    display_height = round(height * scale)

    display_frame = cv2.resize(
        frame,
        (display_width, display_height)
    )

    points = []

    window_name = "Select court: Team A then Team B"

    def mouse_callback(event, x, y, flags, parameter):
        if event == cv2.EVENT_LBUTTONDOWN:
            if len(points) < 8:
                points.append((x, y))

    cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)
    cv2.setMouseCallback(window_name, mouse_callback)

    print("Click 4 floor corners for Team A in perimeter order.")
    print("Then click 4 floor corners for Team B.")
    print("ENTER = save | R = reset | ESC = cancel")

    areas = None

    try:
        while True:
            preview = display_frame.copy()

            for index, point in enumerate(points):
                team = "A" if index < 4 else "B"
                color = TEAM_COLORS[team]

                cv2.circle(
                    preview,
                    point,
                    5,
                    color,
                    -1
                )

                cv2.putText(
                    preview,
                    f"{team}{index % 4 + 1}",
                    (point[0] + 5, point[1] - 5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    color,
                    2
                )

            # Connect the selected corners.
            for start, team in ((0, "A"), (4, "B")):
                selected = points[start:start + 4]

                if len(selected) >= 2:
                    polygon = np.array(
                        selected,
                        dtype=np.int32
                    )

                    cv2.polylines(
                        preview,
                        [polygon],
                        len(selected) == 4,
                        TEAM_COLORS[team],
                        2
                    )

            cv2.putText(
                preview,
                "4 corners A, then 4 corners B | ENTER save | R reset",
                (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (255, 255, 255),
                1
            )

            cv2.imshow(window_name, preview)

            key = cv2.waitKey(30) & 0xFF

            if key == 27:
                raise RuntimeError("Court selection cancelled.")

            if key in (ord("r"), ord("R")):
                points.clear()

            if key == 13 and len(points) == 8:
                # save the corners at the real video size
                source_points = np.array(
                    points,
                    dtype=np.float32
                )

                source_points[:, 0] *= width / display_width
                source_points[:, 1] *= height / display_height

                areas = {
                    "A": source_points[:4],
                    "B": source_points[4:]
                }

                valid = all(
                    cv2.isContourConvex(polygon)
                    and abs(cv2.contourArea(polygon)) > 50
                    for polygon in areas.values()
                )

                if valid:
                    break

                print(
                    "Invalid selection. Press R and click "
                    "each half's corners in perimeter order."
                )

    finally:
        cv2.destroyWindow(window_name)

    settings_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    settings_path.write_text(
        json.dumps(
            {
                "video_info": video_info,
                "areas": {
                    team: polygon.tolist()
                    for team, polygon in areas.items()
                }
            },
            indent=2
        )
    )

    return areas


def assign_team(box, court_areas):
    # player feet decide which half they are on

    x1, y1, x2, y2 = box

    foot_position = (
        float((x1 + x2) / 2),
        float(y2)
    )

    matching_teams = [
        team
        for team, polygon in court_areas.items()
        if cv2.pointPolygonTest(
            polygon,
            foot_position,
            False
        ) >= 0
    ]

    # ignore people outside the court or on both areas
    if len(matching_teams) != 1:
        return None

    return matching_teams[0]


def draw_court_areas(frame, court_areas):
    # draw the court borders

    for team, polygon in court_areas.items():
        cv2.polylines(
            frame,
            [polygon.astype(np.int32)],
            True,
            TEAM_COLORS[team],
            1
        )

## Video output

In [4]:
def create_video_writer(video_cap, output_filename):
    frame_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = float(video_cap.get(cv2.CAP_PROP_FPS))

    if frame_width <= 0 or frame_height <= 0:
        raise ValueError("Cannot read the input video's dimensions.")

    if not math.isfinite(fps) or fps <= 0:
        raise ValueError("Cannot read the input video's frame rate.")

    output_path = Path(output_filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")

    writer = cv2.VideoWriter(
        str(output_path),
        fourcc,
        fps,
        (frame_width, frame_height)
    )

    if not writer.isOpened():
        raise RuntimeError(
            f"Cannot create the output video: {output_path}"
        )

    return writer

## Track the players and ball

Writes the next numbered MP4 and CSV to `outputs/`.

In [5]:
def track_volleyball():
    # output files get a new number each run


    output_folder.mkdir(parents=True, exist_ok=True)

    run_number = 1

    # dont overwrite old results
    while True:
        output_path = (
            output_folder / f"volleyball_tracking_{run_number}.mp4"
        )

        csv_path = (
            output_folder / f"player_tracks_{run_number}.csv"
        )

        if not output_path.exists() and not csv_path.exists():
            break

        run_number += 1



    if not input_path.is_file():
        raise FileNotFoundError(
            f"Input video not found: {input_path}"
        )

    if not ball_model_path.is_file():
        raise FileNotFoundError(
            f"Volleyball model not found: {ball_model_path}"
        )

    output_path.parent.mkdir(parents=True, exist_ok=True)
    player_model_path.parent.mkdir(parents=True, exist_ok=True)
    if not player_model_path.exists():
        # get the standard person model if it isnt here yet
        import urllib.request
        url = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt"
        print("Downloading standard YOLOv8n player model...")
        urllib.request.urlretrieve(url, str(player_model_path))

    # load both models once
    ball_model = YOLO(str(ball_model_path))

    # standard COCO weights go in player_models
    player_model = YOLO(str(player_model_path))

    print("Ball model classes:", ball_model.names)
    print("Player model classes:", player_model.names)

    # only use classes named ball
    ball_class_ids = [
        class_id
        for class_id, name in ball_model.names.items()
        if "ball" in name.lower()
    ]

    if not ball_class_ids:
        raise ValueError(
            "The ball model has no class containing 'ball'. "
            "Check that yolov8n.pt is your volleyball-trained model."
        )

    # make sure this model is for people
    if player_model.names.get(0, "").lower() != "person":
        raise ValueError(
            "The player model is incorrect. "
            "player_models/yolov8n.pt must be the standard "
            "COCO model, not your volleyball-trained model."
        )

    # open the video and load the court
    cap = cv2.VideoCapture(str(input_path))

    if not cap.isOpened():
        raise RuntimeError(
            f"Cannot open the input video: {input_path}"
        )
    court_areas = load_or_select_court(
        cap,
        court_settings_path,
        input_path
    )
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = create_video_writer(cap, output_path)

    # keep track of the ball between frames
    frame_number = 0

    ball_trail = deque(maxlen=10)
    previous_ball = None
    missed_ball_frames = 0

    # reset the ball after 0.3 seconds missing
    max_missed_frames = max(1, round(fps * 0.3))

    # ignore detections too far from the previous ball
    max_ball_jump = (
        0.25 * np.hypot(frame_width, frame_height)
    )

    # text size based on the video
    font_scale = max(0.4, frame_width / 1800)

    # write the video and player data
    try:
        with csv_path.open(
            "w",
            newline="",
            encoding="utf-8"
        ) as csv_file:

            csv_writer = csv.writer(csv_file)

            csv_writer.writerow([
                "frame",
                "time_seconds",
                "player_id",
                "team",
                "confidence",
                "x1",
                "y1",
                "x2",
                "y2"
            ])

            while True:
                success, frame = cap.read()

                if not success:
                    break

                # use the original frame for both models
                annotated_frame = frame.copy()

                time_seconds = frame_number / fps

                # detect players and figure out their team
                player_result = player_model.track(
                    frame,
                    persist=True,
                    tracker="bytetrack.yaml",
                    classes=[0],
                    conf=0.35,
                    verbose=False
                )[0]

                player_boxes = player_result.boxes

                tracked_player_count = 0
                team_counts = {"A": 0, "B": 0}

                draw_court_areas(
                    annotated_frame,
                    court_areas
                )

                if (
                    player_boxes is not None
                    and player_boxes.id is not None
                ):
                    coordinates = (
                        player_boxes.xyxy.cpu().numpy()
                    )

                    player_ids = (
                        player_boxes.id.int().cpu().tolist()
                    )

                    confidences = (
                        player_boxes.conf.cpu().tolist()
                    )

                    for box, player_id, confidence in zip(
                        coordinates,
                        player_ids,
                        confidences
                    ):
                        team = assign_team(
                            box,
                            court_areas
                        )

                        # skip anyone off the court
                        if team is None:
                            continue

                        tracked_player_count += 1
                        team_counts[team] += 1

                        x1, y1, x2, y2 = map(int, box)

                        color = TEAM_COLORS[team]

                        cv2.rectangle(
                            annotated_frame,
                            (x1, y1),
                            (x2, y2),
                            color,
                            2
                        )

                        cv2.putText(
                            annotated_frame,
                            f"Team {team} | ID {player_id}",
                            (x1, max(18, y1 - 6)),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            font_scale,
                            color,
                            1,
                            cv2.LINE_AA
                        )

                        # use the bottom of the box to pick a team
                        foot_position = (
                            int((box[0] + box[2]) / 2),
                            int(box[3])
                        )

                        cv2.circle(
                            annotated_frame,
                            foot_position,
                            4,
                            color,
                            -1
                        )

                        csv_writer.writerow([
                            frame_number,
                            round(time_seconds, 3),
                            player_id,
                            team,
                            round(confidence, 3),
                            x1,
                            y1,
                            x2,
                            y2
                        ])

                # find the ball
                ball_result = ball_model.predict(
                    frame,
                    classes=ball_class_ids,
                    conf=0.5,
                    verbose=False
                )[0]

                candidates = []

                if ball_result.boxes is not None:
                    coordinates = (
                        ball_result.boxes.xyxy.cpu().numpy()
                    )

                    confidences = (
                        ball_result.boxes.conf.cpu().tolist()
                    )

                    for box, confidence in zip(
                        coordinates,
                        confidences
                    ):
                        x1, y1, x2, y2 = box

                        center = np.array([
                            (x1 + x2) / 2,
                            (y1 + y2) / 2
                        ])

                        candidates.append(
                            (center, confidence)
                        )

                # pick which ball to follow
                selected_ball = None

                if candidates:
                    if previous_ball is None:
                        # first ball: go with the highest confidence
                        selected_ball = max(
                            candidates,
                            key=lambda item: item[1]
                        )[0]

                    else:
                        # next ball: pick the closest one
                        closest = min(
                            candidates,
                            key=lambda item: np.linalg.norm(
                                item[0] - previous_ball
                            )
                        )[0]

                        distance = np.linalg.norm(
                            closest - previous_ball
                        )

                        if distance <= max_ball_jump:
                            selected_ball = closest

                # draw the ball trail
                if selected_ball is not None:
                    previous_ball = selected_ball
                    missed_ball_frames = 0

                    center = tuple(
                        int(value) for value in selected_ball
                    )

                    ball_trail.append(center)

                    cv2.circle(
                        annotated_frame,
                        center,
                        8,
                        (255, 0, 0),
                        2
                    )

                    points = list(ball_trail)

                    for start, end in zip(
                        points[:-1],
                        points[1:]
                    ):
                        cv2.line(
                            annotated_frame,
                            start,
                            end,
                            (0, 0, 255),
                            2
                        )

                else:
                    missed_ball_frames += 1

                    # clear the trail if the ball disappears
                    ball_trail.clear()

                    if missed_ball_frames >= max_missed_frames:
                        previous_ball = None

                # show the time and team counts
                cv2.rectangle(
                    annotated_frame,
                    (0, 0),
                    (frame_width, 36),
                    (25, 25, 25),
                    -1
                )

                cv2.putText(
                    annotated_frame,
                    (
                        f"Time: {time_seconds:.1f}s | "
                        f"Team A: {team_counts['A']} | "
                        f"Team B: {team_counts['B']}"
                    ),
                    (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale,
                    (255, 255, 255),
                    1,
                    cv2.LINE_AA
                )

                # save the frame
                writer.write(annotated_frame)

                frame_number += 1

                if frame_number % 50 == 0:
                    print(
                        f"Processed {frame_number} frames",
                        flush=True
                    )

    finally:
        cap.release()
        writer.release()

    print(f"Finished processing {frame_number} frames.")
    print(f"Output video: {output_path}")
    print(f"Player tracking CSV: {csv_path}")

In [9]:
track_volleyball()

Ball model classes: {0: 'ball'}
Player model classes: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: '